In [1]:
# Attack Tree Data Structure Storage and Analysis Module
# This module creates a comprehensive data structure for storing attack tree information

import os, re, json, html
import pandas as pd
from typing import Dict, List, Set, Any, Optional, Tuple
from dataclasses import dataclass, asdict
from datetime import datetime

# Config
DATASET_PATH = "device_compromise_dataset.xlsx"
OUTPUT_DIR = "attack_tree_data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# MITRE tactic ordering
MITRE_TACTIC_ORDER = [
    "Reconnaissance", "Resource Development", "Initial Access", "Execution",
    "Persistence", "Privilege Escalation", "Defense Evasion", "Credential Access",
    "Discovery", "Lateral Movement", "Collection", "Command and Control",
    "Exfiltration", "Impact"
]

@dataclass
class AttackNode:
    """Represents a single node in the attack tree"""
    id: str
    label: str
    node_type: str  # root, tactic, cwe, tech, subtech, capec, gate
    metadata: Dict[str, Any]
    parent_ids: List[str]  # Can have multiple parents (for gates)
    child_ids: List[str]
    tactic_context: Optional[str] = None  # Which tactic this node belongs to
    path_signature: Optional[str] = None  # For deduplication tracking
    created_at: Optional[str] = None
    
    def __post_init__(self):
        if self.created_at is None:
            self.created_at = datetime.now().isoformat()

@dataclass
class AttackPath:
    """Represents a complete attack path from CWE to CAPEC"""
    path_id: str
    cwe: str
    technique: str
    subtechnique: str  # Empty string if direct technique
    capec: str
    tactic: str
    confidence_score: float = 1.0
    path_nodes: List[str] = None  # Node IDs in this path
    
    def __post_init__(self):
        if self.path_nodes is None:
            self.path_nodes = []
    
    def get_signature(self) -> str:
        """Generate unique signature for path deduplication"""
        return f"{self.cwe}→{self.technique}→{self.subtechnique}→{self.capec}"

@dataclass
class AttackTreeStructure:
    """Complete attack tree data structure"""
    metadata: Dict[str, Any]
    nodes: Dict[str, AttackNode]
    paths: Dict[str, AttackPath]
    adjacency_list: Dict[str, List[str]]  # node_id -> [child_node_ids]
    reverse_adjacency: Dict[str, List[str]]  # node_id -> [parent_node_ids]
    tactic_order: List[str]
    deduplication_log: List[Dict[str, Any]]
    statistics: Dict[str, Any]
    created_at: str
    
    def __post_init__(self):
        if not hasattr(self, 'created_at') or self.created_at is None:
            self.created_at = datetime.now().isoformat()

class AttackTreeBuilder:
    """Builds attack tree and stores in comprehensive data structure"""
    
    def __init__(self, dataset_path: str):
        self.dataset_path = dataset_path
        self.df = None
        self.tree_structure = None
        self.seen_paths: Set[str] = set()
        self.deduplication_log: List[Dict[str, Any]] = []
        self.node_counter = 0
        
    def load_data(self):
        """Load and prepare dataset"""
        self.df = pd.read_excel(self.dataset_path, index_col=False)
        self.df = self.df.fillna("")
        
        if "tactics" not in self.df.columns:
            raise KeyError("Expected 'tactics' column in the dataset")
        
        self.df["tactics"] = self.df["tactics"].astype(str).str.strip()
        print(f"Loaded dataset with {len(self.df)} rows")
    
    def _normalize_name(self, s: str) -> str:
        return re.sub(r'[^0-9a-z]', '', str(s).lower())
    
    def extract_node_metadata(self, df_subset: pd.DataFrame, node_type: str) -> Dict[str, Any]:
        """Extract comprehensive metadata for a node"""
        cols_map = {self._normalize_name(c): c for c in df_subset.columns.tolist()}
        
        metadata_keys = {
            "cwe": [
                'CVE ID', 'Description', 'cwe_Name', 'cwe_Abstraction', 'cwe_Structure',
                'cwe_Status', 'cwe_Description', 'cwe_Extended_Description', 
                'cwe_Related_Weaknesses', 'cwe_Weakness_Ordinality', 'cwe_Language_Name',
                'cwe_Technology_Class', 'cwe_Alternate_Terms', 'cwe_Background_Details',
                'cwe_Modes_Of_Introduction', 'cwe_Likelihood_Of_Exploit', 
                'cwe_Common_Consequences', 'cwe_Detection_Methods', 'cwe_Potential_Mitigations',
                'attackVector', 'baseScore', 'baseSeverity', 'attackComplexity',
                'privilegesRequired', 'userInteraction', 'confidentialityImpact',
                'availabilityImpact', 'exploitabilityScore'
            ],
            "capec": [
                'CVE ID', 'capec_Name', 'capec_Mitre_Techniques', 'capec_Abstraction',
                'capec_Status', 'capec_Description', 'capec_Extended_Description',
                'capec_Likelihood_Of_Attack', 'capec_Typical_Severity', 'capec_Execution_Flow',
                'capec_Prerequisites', 'capec_Skills_Required', 'capec_Resources_Required',
                'capec_Consequences', 'capec_Mitigations', 'capec_Example_Instances',
                'capec_Related_Weaknesses', 'capec_Taxonomy_Mappings'
            ],
            "tech": [
                'CVE ID', 'Description', 'tech_description', 'technique_name', 'subtechnique_name',
                'tactics', 'detection', 'platforms', 'data sources', 'defenses bypassed',
                'contributors', 'permissions required', 'supports remote', 'system requirements',
                'impact type', 'effective permissions', 'attackVector', 'baseScore', 'baseSeverity',
                'attackComplexity', 'privilegesRequired', 'userInteraction', 'confidentialityImpact',
                'availabilityImpact', 'exploitabilityScore'
            ],
            "subtech": [
                'subtechnique_name', 'technique_name', 'tactics', 'detection', 'platforms', 'data sources'
            ]
        }
        
        keys = metadata_keys.get(node_type, df_subset.columns.tolist())
        meta = {}
        
        for k in keys:
            nk = self._normalize_name(k)
            colname = cols_map.get(nk)
            if colname and colname in df_subset.columns:
                vals = df_subset[colname].astype(str).map(str.strip)
                vals = [v for v in vals if v != "" and v.lower() != "nan"]
                uniq = []
                for v in vals:
                    if v not in uniq:
                        uniq.append(v)
                if len(uniq) == 1:
                    meta[k] = uniq[0]
                elif len(uniq) > 1:
                    meta[k] = " | ".join(uniq)
        
        # Add analysis metadata
        meta['_analysis'] = {
            'row_count': len(df_subset),
            'unique_values_count': {col: df_subset[col].nunique() for col in df_subset.columns if col in df_subset.columns},
            'extraction_timestamp': datetime.now().isoformat()
        }
        
        return meta
    
    def analyze_cwe_subtree(self, cwe_subset: pd.DataFrame, cwe_name: str, tactic: str) -> Dict[str, Any]:
        """Analyze complete subtree structure for a CWE"""
        structure = {
            'has_valid_paths': False,
            'techniques': {},
            'new_paths': [],
            'duplicate_paths': []
        }
        
        techniques = [x for x in cwe_subset.get("technique_name", pd.Series([], dtype=object))
                      .unique().tolist() if x != ""]
        
        for tech in techniques:
            tech_subset = cwe_subset[cwe_subset["technique_name"] == tech]
            subtechs = [s for s in tech_subset.get("subtechnique_name", pd.Series([], dtype=object))
                        .unique().tolist() if s != ""]
            
            tech_structure = {
                'has_subtechs': len(subtechs) > 0,
                'valid_subtechs': {},
                'direct_capecs': [],
                'has_valid_paths': False
            }
            
            if subtechs:
                # Process subtechniques
                for subtech in subtechs:
                    sub_rows = tech_subset[tech_subset["subtechnique_name"] == subtech]
                    capec_names = [c for c in sub_rows.get("capec_Name", pd.Series([], dtype=object))
                                   .unique().tolist() if c != ""]
                    
                    if not capec_names:
                        capec_candidates = ["CAPEC ?"]
                    else:
                        capec_candidates = capec_names
                    
                    new_capecs = []
                    for cap in capec_candidates:
                        path_sig = f"{cwe_name}→{tech}→{subtech}→{cap}"
                        if path_sig not in self.seen_paths:
                            new_capecs.append(cap)
                            structure['new_paths'].append({
                                'cwe': cwe_name, 'technique': tech, 'subtechnique': subtech,
                                'capec': cap, 'tactic': tactic, 'signature': path_sig
                            })
                        else:
                            structure['duplicate_paths'].append({
                                'cwe': cwe_name, 'technique': tech, 'subtechnique': subtech,
                                'capec': cap, 'tactic': tactic, 'signature': path_sig
                            })
                    
                    if new_capecs:
                        tech_structure['valid_subtechs'][subtech] = new_capecs
                        tech_structure['has_valid_paths'] = True
            else:
                # Process direct technique CAPECs
                capec_names = [c for c in tech_subset.get("capec_Name", pd.Series([], dtype=object))
                               .unique().tolist() if c != ""]
                
                if not capec_names:
                    capec_candidates = ["CAPEC ?"]
                else:
                    capec_candidates = capec_names
                
                new_capecs = []
                for cap in capec_candidates:
                    path_sig = f"{cwe_name}→{tech}→→{cap}"
                    if path_sig not in self.seen_paths:
                        new_capecs.append(cap)
                        structure['new_paths'].append({
                            'cwe': cwe_name, 'technique': tech, 'subtechnique': '',
                            'capec': cap, 'tactic': tactic, 'signature': path_sig
                        })
                    else:
                        structure['duplicate_paths'].append({
                            'cwe': cwe_name, 'technique': tech, 'subtechnique': '',
                            'capec': cap, 'tactic': tactic, 'signature': path_sig
                        })
                
                if new_capecs:
                    tech_structure['direct_capecs'] = new_capecs
                    tech_structure['has_valid_paths'] = True
            
            if tech_structure['has_valid_paths']:
                structure['techniques'][tech] = tech_structure
                structure['has_valid_paths'] = True
        
        return structure
    
    def create_node(self, label: str, node_type: str, metadata: Dict[str, Any], 
                    tactic_context: str = None, path_signature: str = None) -> AttackNode:
        """Create a new attack tree node"""
        node_id = f"{node_type}_{self.node_counter}"
        self.node_counter += 1
        
        return AttackNode(
            id=node_id,
            label=label,
            node_type=node_type,
            metadata=metadata,
            parent_ids=[],
            child_ids=[],
            tactic_context=tactic_context,
            path_signature=path_signature
        )
    
    def add_edge(self, parent_node: AttackNode, child_node: AttackNode, adjacency: Dict, reverse_adj: Dict):
        """Add parent-child relationship"""
        parent_node.child_ids.append(child_node.id)
        child_node.parent_ids.append(parent_node.id)
        
        if parent_node.id not in adjacency:
            adjacency[parent_node.id] = []
        adjacency[parent_node.id].append(child_node.id)
        
        if child_node.id not in reverse_adj:
            reverse_adj[child_node.id] = []
        reverse_adj[child_node.id].append(parent_node.id)
    
    def build_attack_tree_structure(self) -> AttackTreeStructure:
        """Build complete attack tree data structure"""
        if self.df is None:
            self.load_data()
        
        # Initialize data structures
        nodes = {}
        paths = {}
        adjacency_list = {}
        reverse_adjacency = {}
        
        # Determine ordered tactics
        unique_tactics = [t for t in self.df["tactics"].unique() if t != ""]
        ordered_tactics = [t for t in MITRE_TACTIC_ORDER if t in unique_tactics]
        for t in unique_tactics:
            if t not in ordered_tactics:
                ordered_tactics.append(t)
        
        # Create root node
        root_node = self.create_node(
            label="Compromise Device",
            node_type="root",
            metadata={"Description": "Root node of attack tree", "purpose": "Ultimate attack goal"}
        )
        nodes[root_node.id] = root_node
        
        # First pass: Analyze all tactics and CWEs
        valid_tactics_info = []
        tactic_cwe_structures = {}
        
        for i, tactic in enumerate(ordered_tactics):
            tactic_df = self.df[self.df["tactics"] == tactic]
            cwe_names = [x for x in tactic_df.get("cwe_Name", pd.Series([], dtype=object))
                         .unique().tolist() if x != ""]
            
            valid_cwe_structures = {}
            
            for cwe in cwe_names:
                cwe_subset = tactic_df[tactic_df["cwe_Name"] == cwe]
                cwe_structure = self.analyze_cwe_subtree(cwe_subset, cwe, tactic)
                
                if cwe_structure['has_valid_paths']:
                    valid_cwe_structures[cwe] = cwe_structure
                    
                    # Mark paths as seen and create path objects
                    for path_info in cwe_structure['new_paths']:
                        path_sig = path_info['signature']
                        self.seen_paths.add(path_sig)
                        
                        path_obj = AttackPath(
                            path_id=f"path_{len(paths)}",
                            cwe=path_info['cwe'],
                            technique=path_info['technique'],
                            subtechnique=path_info['subtechnique'],
                            capec=path_info['capec'],
                            tactic=path_info['tactic']
                        )
                        paths[path_obj.path_id] = path_obj
                
                # Log duplicates
                for dup_path in cwe_structure['duplicate_paths']:
                    self.deduplication_log.append({
                        'action': 'skipped_duplicate',
                        'path_signature': dup_path['signature'],
                        'tactic': dup_path['tactic'],
                        'timestamp': datetime.now().isoformat()
                    })
            
            if valid_cwe_structures:
                valid_tactics_info.append((i, tactic, tactic_df))
                tactic_cwe_structures[i] = valid_cwe_structures
        
        # Second pass: Build nodes and relationships
        tactic_nodes = []
        
        for i, tactic, tactic_df in valid_tactics_info:
            # Create tactic node
            tactic_node = self.create_node(
                label=tactic,
                node_type="tactic",
                metadata={"tactic": tactic, "mitre_order": i},
                tactic_context=tactic
            )
            nodes[tactic_node.id] = tactic_node
            tactic_nodes.append(tactic_node)
            
            # Add root -> tactic relationship
            self.add_edge(root_node, tactic_node, adjacency_list, reverse_adjacency)
            
            cwe_structures = tactic_cwe_structures[i]
            cwe_nodes = []
            
            # Create CWE nodes
            for cwe_name, cwe_structure in cwe_structures.items():
                cwe_subset = tactic_df[tactic_df["cwe_Name"] == cwe_name]
                cwe_meta = self.extract_node_metadata(cwe_subset, "cwe")
                
                cwe_node = self.create_node(
                    label=cwe_name,
                    node_type="cwe",
                    metadata=cwe_meta,
                    tactic_context=tactic
                )
                nodes[cwe_node.id] = cwe_node
                cwe_nodes.append(cwe_node)
                
                # Add tactic -> cwe relationship  
                self.add_edge(tactic_node, cwe_node, adjacency_list, reverse_adjacency)
                
                # Build technique/subtechnique/CAPEC structure
                for tech_name, tech_structure in cwe_structure['techniques'].items():
                    tech_subset = cwe_subset[cwe_subset["technique_name"] == tech_name]
                    
                    if tech_structure['has_subtechs']:
                        # Process subtechniques
                        for subtech_name, capec_list in tech_structure['valid_subtechs'].items():
                            sub_rows = tech_subset[tech_subset["subtechnique_name"] == subtech_name]
                            sub_meta = self.extract_node_metadata(sub_rows, "subtech")
                            
                            subtech_node = self.create_node(
                                label=subtech_name,
                                node_type="subtech",
                                metadata=sub_meta,
                                tactic_context=tactic
                            )
                            nodes[subtech_node.id] = subtech_node
                            
                            # Create AND gate: CWE -> (subtech AND capecs)
                            and_gate = self.create_node(
                                label="AND",
                                node_type="gate",
                                metadata={"gate_type": "AND", "purpose": "Require both weakness and attack pattern"},
                                tactic_context=tactic
                            )
                            nodes[and_gate.id] = and_gate
                            
                            self.add_edge(cwe_node, and_gate, adjacency_list, reverse_adjacency)
                            self.add_edge(and_gate, subtech_node, adjacency_list, reverse_adjacency)
                            
                            # Create CAPEC nodes and OR gate
                            capec_nodes = []
                            for cap in capec_list:
                                if cap == "CAPEC ?":
                                    cap_meta = {"capec_Name": "CAPEC ?", "Description": "Unknown CAPEC pattern"}
                                else:
                                    cap_rows = sub_rows[sub_rows["capec_Name"] == cap]
                                    cap_meta = self.extract_node_metadata(cap_rows, "capec")
                                
                                capec_node = self.create_node(
                                    label=cap,
                                    node_type="capec",
                                    metadata=cap_meta,
                                    tactic_context=tactic,
                                    path_signature=f"{cwe_name}→{tech_name}→{subtech_name}→{cap}"
                                )
                                nodes[capec_node.id] = capec_node
                                capec_nodes.append(capec_node)
                                
                                # Update path with node IDs
                                for path_obj in paths.values():
                                    if (path_obj.cwe == cwe_name and path_obj.technique == tech_name and 
                                        path_obj.subtechnique == subtech_name and path_obj.capec == cap):
                                        path_obj.path_nodes = [cwe_node.id, and_gate.id, subtech_node.id, capec_node.id]
                                        break
                            
                            if len(capec_nodes) > 1:
                                or_gate = self.create_node(
                                    label="OR",
                                    node_type="gate", 
                                    metadata={"gate_type": "OR", "purpose": "Alternative attack patterns"},
                                    tactic_context=tactic
                                )
                                nodes[or_gate.id] = or_gate
                                self.add_edge(and_gate, or_gate, adjacency_list, reverse_adjacency)
                                
                                for capec_node in capec_nodes:
                                    self.add_edge(or_gate, capec_node, adjacency_list, reverse_adjacency)
                            else:
                                self.add_edge(and_gate, capec_nodes[0], adjacency_list, reverse_adjacency)
                    
                    else:
                        # Process direct technique CAPECs
                        capec_list = tech_structure['direct_capecs']
                        if capec_list:
                            tech_meta = self.extract_node_metadata(tech_subset, "tech")
                            
                            tech_node = self.create_node(
                                label=tech_name,
                                node_type="tech",
                                metadata=tech_meta,
                                tactic_context=tactic
                            )
                            nodes[tech_node.id] = tech_node
                            
                            # Create AND gate: CWE -> (tech AND capecs)
                            and_gate = self.create_node(
                                label="AND",
                                node_type="gate",
                                metadata={"gate_type": "AND", "purpose": "Require both weakness and attack pattern"},
                                tactic_context=tactic
                            )
                            nodes[and_gate.id] = and_gate
                            
                            self.add_edge(cwe_node, and_gate, adjacency_list, reverse_adjacency)
                            self.add_edge(and_gate, tech_node, adjacency_list, reverse_adjacency)
                            
                            # Create CAPEC nodes
                            capec_nodes = []
                            for cap in capec_list:
                                if cap == "CAPEC ?":
                                    cap_meta = {"capec_Name": "CAPEC ?", "Description": "Unknown CAPEC pattern"}
                                else:
                                    cap_rows = tech_subset[tech_subset["capec_Name"] == cap]
                                    cap_meta = self.extract_node_metadata(cap_rows, "capec")
                                
                                capec_node = self.create_node(
                                    label=cap,
                                    node_type="capec",
                                    metadata=cap_meta,
                                    tactic_context=tactic,
                                    path_signature=f"{cwe_name}→{tech_name}→→{cap}"
                                )
                                nodes[capec_node.id] = capec_node
                                capec_nodes.append(capec_node)
                                
                                # Update path with node IDs
                                for path_obj in paths.values():
                                    if (path_obj.cwe == cwe_name and path_obj.technique == tech_name and 
                                        path_obj.subtechnique == '' and path_obj.capec == cap):
                                        path_obj.path_nodes = [cwe_node.id, and_gate.id, tech_node.id, capec_node.id]
                                        break
                            
                            if len(capec_nodes) > 1:
                                or_gate = self.create_node(
                                    label="OR",
                                    node_type="gate",
                                    metadata={"gate_type": "OR", "purpose": "Alternative attack patterns"},
                                    tactic_context=tactic
                                )
                                nodes[or_gate.id] = or_gate
                                self.add_edge(and_gate, or_gate, adjacency_list, reverse_adjacency)
                                
                                for capec_node in capec_nodes:
                                    self.add_edge(or_gate, capec_node, adjacency_list, reverse_adjacency)
                            else:
                                self.add_edge(and_gate, capec_nodes[0], adjacency_list, reverse_adjacency)
        
        # Calculate statistics
        node_type_counts = {}
        for node in nodes.values():
            node_type_counts[node.node_type] = node_type_counts.get(node.node_type, 0) + 1
        
        statistics = {
            'total_nodes': len(nodes),
            'total_paths': len(paths),
            'total_edges': sum(len(children) for children in adjacency_list.values()),
            'node_type_counts': node_type_counts,
            'tactic_count': len([n for n in nodes.values() if n.node_type == 'tactic']),
            'unique_paths_count': len(self.seen_paths),
            'duplicate_paths_skipped': len(self.deduplication_log),
            'tactics_processed': len(valid_tactics_info),
            'build_timestamp': datetime.now().isoformat()
        }
        
        # Create tree structure
        self.tree_structure = AttackTreeStructure(
            metadata={
                'title': 'Device Compromise Attack Tree',
                'description': 'Complete attack tree with deduplication and path analysis',
                'dataset_source': self.dataset_path,
                'build_method': 'complete_subtree_deduplication',
                'mitre_attack_version': 'latest'
            },
            nodes=nodes,
            paths=paths,
            adjacency_list=adjacency_list,
            reverse_adjacency=reverse_adjacency,
            tactic_order=ordered_tactics,
            deduplication_log=self.deduplication_log,
            statistics=statistics,
            created_at=datetime.now().isoformat()
        )
        
        return self.tree_structure
    
    def save_structure(self, filename: str = None):
        """Save tree structure to JSON files"""
        if self.tree_structure is None:
            raise ValueError("Tree structure not built. Call build_attack_tree_structure() first.")
        
        if filename is None:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            filename = f"attack_tree_structure_{timestamp}"
        
        # Convert dataclasses to dictionaries for JSON serialization
        structure_dict = {
            'metadata': self.tree_structure.metadata,
            'nodes': {k: asdict(v) for k, v in self.tree_structure.nodes.items()},
            'paths': {k: asdict(v) for k, v in self.tree_structure.paths.items()},
            'adjacency_list': self.tree_structure.adjacency_list,
            'reverse_adjacency': self.tree_structure.reverse_adjacency,
            'tactic_order': self.tree_structure.tactic_order,
            'deduplication_log': self.tree_structure.deduplication_log,
            'statistics': self.tree_structure.statistics,
            'created_at': self.tree_structure.created_at
        }
        
        # Save complete structure
        complete_path = os.path.join(OUTPUT_DIR, f"{filename}_complete.json")
        with open(complete_path, 'w', encoding='utf-8') as f:
            json.dump(structure_dict, f, indent=2, ensure_ascii=False)
        
        # Save nodes only (for analysis)
        nodes_path = os.path.join(OUTPUT_DIR, f"{filename}_nodes.json") 
        with open(nodes_path, 'w', encoding='utf-8') as f:
            json.dump({k: asdict(v) for k, v in self.tree_structure.nodes.items()}, f, indent=2, ensure_ascii=False)
        
        # Save paths only (for analysis)
        paths_path = os.path.join(OUTPUT_DIR, f"{filename}_paths.json")
        with open(paths_path, 'w', encoding='utf-8') as f:
            json.dump({k: asdict(v) for k, v in self.tree_structure.paths.items()}, f, indent=2, ensure_ascii=False)
        
        # Save statistics and metadata
        stats_path = os.path.join(OUTPUT_DIR, f"{filename}_stats.json")
        with open(stats_path, 'w', encoding='utf-8') as f:
            json.dump({
                'metadata': self.tree_structure.metadata,
                'statistics': self.tree_structure.statistics,
                'deduplication_log': self.tree_structure.deduplication_log
            }, f, indent=2, ensure_ascii=False)
        
        print(f"Attack tree structure saved:")
        print(f"  Complete: {complete_path}")
        print(f"  Nodes only: {nodes_path}")
        print(f"  Paths only: {paths_path}")
        print(f"  Statistics: {stats_path}")
        
        return {
            'complete': complete_path,
            'nodes': nodes_path, 
            'paths': paths_path,
            'statistics': stats_path
        }

class AttackTreeAnalyzer:
    """Analysis utilities for attack tree structures"""
    
    def __init__(self, tree_structure: AttackTreeStructure):
        self.tree = tree_structure
    
    def get_attack_paths_by_tactic(self) -> Dict[str, List[AttackPath]]:
        """Group attack paths by tactic"""
        paths_by_tactic = {}
        for path in self.tree.paths.values():
            if path.tactic not in paths_by_tactic:
                paths_by_tactic[path.tactic] = []
            paths_by_tactic[path.tactic].append(path)
        return paths_by_tactic
    
    def get_cwe_coverage(self) -> Dict[str, Any]:
        """Analyze CWE coverage across tactics"""
        cwe_info = {}
        for node in self.tree.nodes.values():
            if node.node_type == 'cwe':
                cwe_name = node.label
                if cwe_name not in cwe_info:
                    cwe_info[cwe_name] = {
                        'tactics': [],
                        'techniques': set(),
                        'capecs': set(),
                        'node_ids': []
                    }
                cwe_info[cwe_name]['tactics'].append(node.tactic_context)
                cwe_info[cwe_name]['node_ids'].append(node.id)
        
        # Convert sets to lists for JSON serialization
        for cwe_data in cwe_info.values():
            cwe_data['techniques'] = list(cwe_data['techniques'])
            cwe_data['capecs'] = list(cwe_data['capecs'])
        
        return cwe_info
    
    def find_critical_paths(self, criteria: str = 'high_cvss') -> List[AttackPath]:
        """Find critical attack paths based on criteria"""
        critical_paths = []
        
        for path in self.tree.paths.values():
            is_critical = False
            
            if criteria == 'high_cvss':
                # Look for high CVSS scores in path nodes
                for node_id in path.path_nodes:
                    if node_id in self.tree.nodes:
                        node = self.tree.nodes[node_id]
                        if 'baseScore' in node.metadata:
                            try:
                                score = float(node.metadata['baseScore'])
                                if score >= 7.0:  # High or Critical
                                    is_critical = True
                                    break
                            except (ValueError, TypeError):
                                pass
            
            if is_critical:
                critical_paths.append(path)
        
        return critical_paths
    
    def get_node_relationships(self, node_id: str) -> Dict[str, Any]:
        """Get detailed relationship information for a node"""
        if node_id not in self.tree.nodes:
            return {}
        
        node = self.tree.nodes[node_id]
        
        return {
            'node': asdict(node),
            'parents': [asdict(self.tree.nodes[pid]) for pid in node.parent_ids if pid in self.tree.nodes],
            'children': [asdict(self.tree.nodes[cid]) for cid in node.child_ids if cid in self.tree.nodes],
            'paths_involving_node': [asdict(path) for path in self.tree.paths.values() 
                                     if node_id in path.path_nodes]
        }

# Usage example and testing
if __name__ == "__main__":
    print("Building Attack Tree Data Structure...")
    
    # Build tree structure
    builder = AttackTreeBuilder(DATASET_PATH)
    tree_structure = builder.build_attack_tree_structure()
    
    # Save structure
    file_paths = builder.save_structure()
    
    # Print summary
    print(f"\nAttack Tree Summary:")
    print(f"Total nodes: {tree_structure.statistics['total_nodes']}")
    print(f"Total attack paths: {tree_structure.statistics['total_paths']}")
    print(f"Total edges: {tree_structure.statistics['total_edges']}")
    print(f"Node types: {tree_structure.statistics['node_type_counts']}")
    print(f"Tactics processed: {tree_structure.statistics['tactics_processed']}")
    print(f"Duplicate paths skipped: {tree_structure.statistics['duplicate_paths_skipped']}")
    
    # Analyze tree
    analyzer = AttackTreeAnalyzer(tree_structure)
    
    # Get paths by tactic
    paths_by_tactic = analyzer.get_attack_paths_by_tactic()
    print(f"\nPaths by tactic:")
    for tactic, paths in paths_by_tactic.items():
        print(f"  {tactic}: {len(paths)} paths")
    
    # Get CWE coverage
    cwe_coverage = analyzer.get_cwe_coverage()
    print(f"\nCWE coverage: {len(cwe_coverage)} unique CWEs")
    
    # Find critical paths
    critical_paths = analyzer.find_critical_paths()
    print(f"Critical paths (high CVSS): {len(critical_paths)}")
    
    print(f"\nData structure files created in: {OUTPUT_DIR}/")
    print("✓ Complete tree structure with full metadata")
    print("✓ Ready for tree reconstruction and analysis")
    print("✓ JSON format for easy integration with other tools")

Building Attack Tree Data Structure...
Loaded dataset with 88 rows
Attack tree structure saved:
  Complete: attack_tree_data/attack_tree_structure_20250913_025539_complete.json
  Nodes only: attack_tree_data/attack_tree_structure_20250913_025539_nodes.json
  Paths only: attack_tree_data/attack_tree_structure_20250913_025539_paths.json
  Statistics: attack_tree_data/attack_tree_structure_20250913_025539_stats.json

Attack Tree Summary:
Total nodes: 100
Total attack paths: 27
Total edges: 99
Node types: {'root': 1, 'tactic': 8, 'cwe': 15, 'subtech': 12, 'gate': 26, 'capec': 27, 'tech': 11}
Tactics processed: 8
Duplicate paths skipped: 24

Paths by tactic:
  Initial Access: 1 paths
  Persistence: 8 paths
  Privilege Escalation: 5 paths
  Defense Evasion: 3 paths
  Credential Access: 6 paths
  Discovery: 1 paths
  Lateral Movement: 1 paths
  Collection: 2 paths

CWE coverage: 5 unique CWEs
Critical paths (high CVSS): 25

Data structure files created in: attack_tree_data/
✓ Complete tree st